<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-04-rag/lesson-4.5-context-engineering/notebooks/GCP_Capstone_4.5_Context_Engineering.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.5 Context Engineering — Hybrid Retrieval, Reranking, Budgeted Packing & Explicit Caching
**Netsetos GenAI Engineering — GCP Capstone** · Module 4 · new in v1.1

The context window is a budget, not a bucket. In this notebook DocuMind's 4.2 pipeline gains:

- a **TokenBudget** for every part of a request (system · tenant pack · chunks · history · answer)
- **hybrid retrieval**: Firestore dense search + BM25 fused with Reciprocal Rank Fusion
- **reranking** with the Discovery Engine Rank API (top 5 of 20 candidates)
- **budgeted packing** that never loses a citation's source/page metadata
- an **explicit per-tenant Gemini cache** recorded in Firestore `tenant_caches`
- a **dual-priced INR cost line** (introductory to 2026-12-31, standard from 2027-01-01)

Core path: Colab on the Module 4 assets (Firestore `chunks`). Production lane: the same modules inside `deploy/services/rag-api/` (requires 12.1–12.5 in v1.1 for rag-api, 12.8 in v2.0 for documind-chat).

## Setup

Two `genai.Client` objects, always: `global` for Gemini 3.x generation and explicit caches, `us-central1` for `text-embedding-005`. Set `PROJECT_ID`.

In [ ]:
!pip install -q google-genai==2.21.0 google-cloud-firestore==2.30.0 google-cloud-discoveryengine==0.13.11 rank-bm25==0.2.2 pydantic==2.13.5
# uv equivalent (course-stack convention):
#   uv add google-genai==2.21.0 google-cloud-firestore==2.30.0 google-cloud-discoveryengine==0.13.11 rank-bm25==0.2.2 pydantic==2.13.5
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
TENANT = 'acme'
USD_INR = 85

import json, os, re, subprocess, time, datetime as dt
from dataclasses import dataclass
from typing import List, Literal, Optional
from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud.firestore_v1.base_query import FieldFilter
from google.cloud import discoveryengine_v1 as discoveryengine
from rank_bm25 import BM25Okapi
from pydantic import BaseModel, Field

# Two clients, always. Gemini 3.x generation lives on `global`; embeddings
# (text-embedding-005) are regional-only. An explicit cache lives where generation does -
# `global` (Google's own Gemini 3 caching sample; CLAUDE.md, 2026-09-04) - so it is created
# and used through gen_client. There is no regional path for a Gemini 3.x cache.
gen_client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')
emb_client = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')
db = firestore.Client(project=PROJECT_ID)

# The Rank API (Discovery Engine) must be enabled once per project.
subprocess.run(['gcloud', 'services', 'enable', 'discoveryengine.googleapis.com',
                '--project', PROJECT_ID], capture_output=True, text=True)
print('Clients ready for', PROJECT_ID)

In [ ]:
import json
from pydantic import ValidationError

def draft_of(r, schema, quote_limit=200):
    """The model's structured answer, or a clear error - never a silent None.

    The SDK sets r.parsed to None on ANY validation failure. The first live run of the lane
    (7 Sept 2026) met the one that matters: a quote longer than the contract's 200 characters -
    a statute provision is one sentence - which threw fourteen right answers away and, worse,
    had been scored as the model refusing. So: read the JSON, trim the quote to the contract,
    validate again; anything else is an error you can read, not a refusal."""
    if r.parsed is not None:
        return r.parsed if isinstance(r.parsed, schema) else schema.model_validate(r.parsed)
    cand = (r.candidates or [None])[0]
    reason = getattr(getattr(cand, "finish_reason", None), "name", "NO_CANDIDATES")
    try:
        obj = json.loads(r.text or "")
    except ValueError:
        raise RuntimeError(f"no JSON to parse (finish_reason={reason}) - raise max_output_tokens if it is MAX_TOKENS")
    if isinstance(obj, dict):
        for c in obj.get("citations") or []:
            if isinstance(c, dict) and isinstance(c.get("quote"), str) and len(c["quote"]) > quote_limit:
                c["quote"] = c["quote"][: quote_limit - 3].rstrip() + "..."
    try:
        return schema.model_validate(obj)
    except ValidationError as e:
        err = e.errors()[0]
        raise RuntimeError(f"the draft failed the contract at {'.'.join(str(x) for x in err['loc'])}: {err['msg']}") from None


### Tenant-filtered vector index + THE corpus

Lesson 4.2 built a vector-only index. Tenant isolation needs `tenant_id` and `embedding` in **one** composite index so `find_nearest` can pre-filter by tenant.

The data is DocuMind's corpus, not a list typed into the cell: `deploy/evals/corpus/acme` — the tenant's synthetic handbook (ten policy clauses such as `NP-03` and `EXP-12`, plus 270 boilerplate sections), its MSA, invoice and annual report, and **thirteen real documents** (`deploy/evals/real_sources.json`). The loader is `deploy/shared/documind_corpus.py` pasted verbatim, the same one 4.2 runs, so this notebook works alone or after 4.1/4.2 and never writes a document twice. `ACME_POLICIES` — the tuple list the tenant pack indexes in Cell 6 — is derived from those chunks, so the pack and the corpus cannot disagree.


In [ ]:
# Firestore (default) DB + a TENANT-FILTERED vector index on chunks (idempotent).
# 4.2 built a vector-only index. Tenant isolation needs tenant_id + embedding in ONE index
# so find_nearest can PRE-filter by tenant. Never post-filter tenants in Python in production.
if '(default)' not in subprocess.run(
        ['gcloud', 'firestore', 'databases', 'list', '--project', PROJECT_ID, '--format=value(name)'],
        capture_output=True, text=True).stdout:
    subprocess.run(['gcloud', 'firestore', 'databases', 'create',
                    '--location=asia-south1', '--project', PROJECT_ID], check=False)

_idx = subprocess.run(
    "gcloud firestore indexes composite create --project=" + PROJECT_ID +
    " --collection-group=chunks --query-scope=COLLECTION"
    " --field-config=field-path=tenant_id,order=ascending"
    " --field-config=field-path=embedding,vector-config='{\"dimension\":\"768\",\"flat\":\"{}\"}'",
    shell=True, capture_output=True, text=True)
_out = (_idx.stdout + _idx.stderr).lower()
print('tenant_id + embedding index:',
      'creating (~2-5 min)' if _idx.returncode == 0
      else ('already exists' if 'already exists' in _out else _idx.stderr.strip()[:90]))
# The ledger's index (12.5, firestore_indexes.tf's second vector index): tenant_id + current + embedding, so
# find_nearest can PRE-filter to the current version of every document. Retired rows are never candidates.
_idx2 = subprocess.run(
    "gcloud firestore indexes composite create --project=" + PROJECT_ID +
    " --collection-group=chunks --query-scope=COLLECTION"
    " --field-config=field-path=tenant_id,order=ascending"
    " --field-config=field-path=current,order=ascending"
    " --field-config=field-path=embedding,vector-config='{\"dimension\":\"768\",\"flat\":\"{}\"}'",
    shell=True, capture_output=True, text=True)
_out2 = (_idx2.stdout + _idx2.stderr).lower()
print('tenant_id + current + embedding index:',
      'creating (~2-5 min)' if _idx2.returncode == 0
      else ('already exists' if 'already exists' in _out2 else _idx2.stderr.strip()[:90]))

# THE corpus, through the kit's loader: deploy/shared/documind_corpus.py pasted VERBATIM
# (tools/check_contract.py holds this cell to it) - the same cell 4.2 runs. ACME's synthetic
# handbook, MSA, invoice and report, and thirteen REAL documents (deploy/evals/
# real_sources.json): about 1,600 chunks, $0.05 of embeddings on a first run, nothing on a re-run,
# and a document the tenant already holds (4.1's Document AI chunks) is never written twice.
# A handbook clause such as "NP-03 - Notice period" is one chunk with the code in its id
# (acme:hr_policy_2026#NP-03): the string the BM25 half of hybrid retrieval is there to find.
# --- kit: begin ---------------------------------------------------------------
import hashlib
KIT_REPO = "https://github.com/netsetos/agentic-ai-weekend-gcp-learners"   # the learner repo carries the kit under deploy/
KIT_BRANCH = "main"   # the learner repo (public): the notebooks and the kit, deploy/, on its main branch
CHUNK_CHARS, CHUNK_OVERLAP = 2000, 200                     # services/ingest/main.py
EMBEDDING_MODEL, EMBEDDING_VERSION = "text-embedding-005", "1"   # stamped on every row; the worker reads the same pair from its environment (variables.tf)
RETENTION_DAYS = 30                                        # a retired row expires this long after it is superseded (the TTL policy in firestore_indexes.tf)
SCHEMA_VERSION = 2                                         # the row shape: doc_key/current (1); chunk_hash, locator, the embedding stamp, expire_at (2)
_SECTION = re.compile(r"^## +(.+?) *$", re.M)
_CODE = re.compile(r"^([A-Z][A-Z0-9]{0,7}(?:-[A-Z0-9]{1,6}){1,2})\b")   # NP-03, IT-SEC-04, MSA-04, GEN-014
_EFFECTIVE = re.compile(r"effective[ _-]?(?:from|date)?\s*[:=]\s*(\d{4}-\d{2}-\d{2})", re.I)   # services/ingest/contracts.py


def find_kit(start: str = ".") -> str:
    """The deploy/evals directory: beside the notebook, above it, or a clone under /content."""
    here = os.path.abspath(start)
    for _ in range(6):
        for cand in (os.path.join(here, "deploy", "evals"), os.path.join(here, "evals"), here):
            if os.path.isfile(os.path.join(cand, "manifest.json")) and os.path.isdir(os.path.join(cand, "corpus")):
                return cand
        here = os.path.dirname(here)
    clone = "/content/agentic-ai-weekend-gcp-learners"
    if not os.path.isdir(clone):
        subprocess.run(["git", "clone", "--depth", "1", "-b", KIT_BRANCH, KIT_REPO, clone], check=True)
    return os.path.join(clone, "deploy", "evals")


def load_documents(tenant: str, evals_dir: str, project_id: str) -> list:
    """Every document of one tenant that has text on disk: the synthetic .md files and the real
    Acts' pypdf mirrors. A scanned PDF with no mirror (posh_act_2013) is skipped - that one is
    lesson 4.1's, and only Document AI can read it. Each carries its VERSION - the sha256 of the
    mirror's bytes, the worker's doc_key for the same bytes - and the date it declares, if any."""
    docs = []
    for m in json.load(open(os.path.join(evals_dir, "manifest.json"), encoding="utf-8")):
        if m["tenant_id"] != tenant or not m.get("chars"):
            continue
        mirror = os.path.join(evals_dir, m["file"].rsplit(".", 1)[0] + ".md")
        if not os.path.isfile(mirror):
            continue
        raw = open(mirror, "rb").read()
        text = raw.decode("utf-8")
        dated = _EFFECTIVE.search(text[:3000])
        docs.append({"slug": m["slug"], "doc_type": m["doc_type"],
                     "source_uri": m["gcs_uri"].replace("${PROJECT_ID}", project_id),
                     "text": text, "sha256": hashlib.sha256(raw).hexdigest(),
                     "effective_from": dated.group(1) if dated else None})
    return docs


def windows(text: str) -> list:
    """The worker's chunker: fixed windows with overlap, ending on a sentence when one is nearby."""
    text = text.strip()
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + CHUNK_CHARS)
        if end < len(text):
            cut = text.rfind(". ", start + CHUNK_CHARS // 2, end)
            if cut != -1:
                end = cut + 1
        piece = text[start:end].strip()
        if piece:
            out.append(piece)
        if end >= len(text):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)
    return out


def chunk_hash(text: str) -> str:
    """The chunk's identity across versions (services/ingest/contracts.py): the hash of its text with the
    whitespace collapsed. A re-wrapped paragraph is the same paragraph; a changed figure is a new chunk."""
    return hashlib.sha256(re.sub(r"\s+", " ", text).strip().encode("utf-8")).hexdigest()


def chunk_document(doc: dict, tenant: str, version: str = "") -> list:
    """Canonical chunk documents - the fields services/ingest/indexer.py writes - minus the embedding. Each carries
    its LOCATOR (the clause code or the section's ordinal; a page and its window for a PDF mirror; `preamble` for
    the text above a handbook's first heading) and its chunk_hash. The first version of a document keeps the classic
    id (acme:hr_policy_2026#NP-03); a re-issue carries its version in the id (acme:hr_policy_2026@1f3a9c2b#NP-03),
    so the retired rows stay beside the current ones and nothing is overwritten. An anchor matches either."""
    text = re.sub(r"\A\s*<!--.*?-->\s*", "", doc["text"], count=1, flags=re.S)   # a mirror's provenance header
    base = {"tenant_id": tenant, "source_uri": doc["source_uri"], "doc_type": doc["doc_type"], "kind": "text"}
    at = f"@{version}" if version else ""
    out = []
    heads = list(_SECTION.finditer(text))
    if heads:                                                  # a handbook: one chunk per section
        for k, piece in enumerate(windows(text[:heads[0].start()])):
            loc = "preamble" + (f"-{k}" if k else "")
            out.append({**base, "chunk_id": f"{tenant}:{doc['slug']}{at}#{loc}", "text": piece, "page_start": 1,
                        "section": None, "locator": loc, "chunk_hash": chunk_hash(piece)})
        for n, h in enumerate(heads):
            body = text[h.end(): heads[n + 1].start() if n + 1 < len(heads) else len(text)].strip()
            title = h.group(1).strip()
            code = _CODE.match(title)
            key = code.group(1) if code else f"s{n + 1}"
            for k, piece in enumerate(windows(f"{title}\n{body}")):
                loc = key + (f"-{k}" if k else "")
                out.append({**base, "chunk_id": f"{tenant}:{doc['slug']}{at}#{loc}", "text": piece, "page_start": 1,
                            "section": title, "locator": loc, "chunk_hash": chunk_hash(piece)})
    else:                                                      # a PDF mirror: pages split by \f
        for p, page in enumerate(text.split("\f"), 1):
            for k, piece in enumerate(windows(page)):
                loc = f"p{p}-{k}"
                out.append({**base, "chunk_id": f"{tenant}:{doc['slug']}{at}#{loc}", "text": piece, "page_start": p,
                            "locator": loc, "chunk_hash": chunk_hash(piece)})
    return out


EMBED_BATCH, EMBED_TOKENS, CHARS_PER_TOKEN = 250, 15_000, 3


def embed_batches(texts: list) -> list:
    """Batches of at most EMBED_BATCH texts AND about EMBED_TOKENS tokens. text-embedding-005 takes
    250 texts per request and 20,000 tokens across them, and a request over either limit fails
    whole; a two-thousand-character chunk is ~500 tokens, so 250 of them are ~125,000. The first
    live corpus load (6 Sept 2026) failed every long Act exactly here - the same rule now lives in
    services/ingest/indexer.py."""
    out, cur, cur_tokens = [], [], 0
    for t in texts:
        tokens = max(1, len(t) // CHARS_PER_TOKEN)
        if cur and (len(cur) >= EMBED_BATCH or cur_tokens + tokens > EMBED_TOKENS):
            out.append(cur); cur, cur_tokens = [], 0
        cur.append(t); cur_tokens += tokens
    if cur:
        out.append(cur)
    return out


def rows_of(db, tenant: str, source_uri: str, collection: str = "chunks") -> list:
    """Every row the tenant holds for one source - its version, its flags, its hash, its vector and its embedding
    stamp. One query, two equality filters, no composite index."""
    from google.cloud.firestore_v1.base_query import FieldFilter
    out = []
    for d in (db.collection(collection).where(filter=FieldFilter("tenant_id", "==", tenant))
                .where(filter=FieldFilter("source_uri", "==", source_uri)).stream()):
        x = d.to_dict() or {}
        out.append({"id": d.id, "ref": d.reference, "doc_key": x.get("doc_key"), "current": x.get("current"),
                    "staged": x.get("staged"), "chunk_hash": x.get("chunk_hash"), "embedding": x.get("embedding"),
                    "embedding_model": x.get("embedding_model"), "embedding_version": x.get("embedding_version")})
    return out


def swap_versions(db, rows: list, new_doc_key: str, retention_days: int = RETENTION_DAYS, effective_to: str = None) -> dict:
    """Visibility is a swap (services/ingest/idempotency.py, the same rule). Every row of new_doc_key becomes current -
    a staged re-issue, or the retired rows of a version uploaded again (the undo) - and then every OTHER current row
    of the source is retired: a flag, superseded_by, and expire_at = now + retention_days for the TTL policy. Never a
    delete. Batches of 400; a reader between two batches sees the new version only (the newest-per-source guard)."""
    import datetime
    from google.cloud import firestore
    expire = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=retention_days)
    activated, retired, n = 0, 0, 0
    batch = db.batch()
    for r in rows:
        if r["doc_key"] == new_doc_key and r["current"] is not True:
            fields = {"current": True, "staged": firestore.DELETE_FIELD, "expire_at": firestore.DELETE_FIELD,
                      "superseded_by": firestore.DELETE_FIELD, "superseded_at": firestore.DELETE_FIELD,
                      "effective_to": firestore.DELETE_FIELD}
            if not r.get("staged"):
                fields["reactivated_at"] = firestore.SERVER_TIMESTAMP     # the undo: a retired row, current again
            batch.update(r["ref"], fields)
            activated += 1; n += 1
        elif r["doc_key"] != new_doc_key and r["current"] is not False:
            fields = {"current": False, "superseded_by": new_doc_key, "superseded_at": firestore.SERVER_TIMESTAMP,
                      "expire_at": expire}
            if effective_to:
                fields["effective_to"] = effective_to
            batch.update(r["ref"], fields)
            retired += 1; n += 1
        if n and n % 400 == 0:                                   # a Firestore batch holds 500 writes
            batch.commit(); batch = db.batch()
    if n % 400:
        batch.commit()
    return {"activated": activated, "retired": retired}


def record_version(db, tenant: str, source_uri: str, doc_key: str, sha256: str, chunks: int, reused: int,
                   embedded: int, retired: int, effective_from: str = None) -> str:
    """The ledger row - sources/{tenant~name}: which version is current and what the reindex cost - and the tenant's
    corpus fingerprint, ledger/{tenant}: the hash of its current doc_keys, the key rag-api's cache follows. What the
    worker writes on every ingest (services/ingest/idempotency.py: record_source, refresh_fingerprint), from a notebook."""
    from google.cloud import firestore
    from google.cloud.firestore_v1.base_query import FieldFilter
    name = source_uri.split("/", 3)[-1]                        # gs://bucket/<tenant>/<file> -> <tenant>/<file>
    db.collection("sources").document(name.replace("/", "~")).set(
        {"tenant_id": tenant, "name": name, "gcs_uri": source_uri, "doc_key": doc_key, "generation": "notebook",
         "sha256": sha256, "chunks": chunks, "reused": reused, "embedded": embedded, "retired": retired,
         "effective_from": effective_from, "status": "indexed", "embedding_model": EMBEDDING_MODEL,
         "embedding_version": EMBEDDING_VERSION, "indexed_at": firestore.SERVER_TIMESTAMP}, merge=True)
    keys = sorted((s.to_dict() or {}).get("doc_key") or "" for s in
                  db.collection("sources").where(filter=FieldFilter("tenant_id", "==", tenant))
                    .where(filter=FieldFilter("status", "==", "indexed")).stream())
    fp = hashlib.sha256("\n".join(keys).encode("utf-8")).hexdigest()[:16]
    db.collection("ledger").document(tenant).set(
        {"tenant_id": tenant, "fingerprint": fp, "versions": len(keys), "last_event": "notebook_seed",
         "updated_at": firestore.SERVER_TIMESTAMP}, merge=True)
    return fp


def seed(db, embed, tenant: str, project_id: str, evals_dir: str = None, collection: str = "chunks",
         retention_days: int = RETENTION_DAYS) -> dict:
    """Write one tenant's corpus into Firestore, version-aware and idempotent (12 September 2026).

    A document's version is the hash of its bytes - the worker's doc_key, tenant_sha256, for the same bytes.
    The tenant holds this version, current: nothing to do. Holds it retired (a later version replaced it):
    the UNDO - its rows come back current, the later version is retired, nothing is embedded. Holds another
    version: the RE-ISSUE - the new chunks are written staged, every unchanged chunk's vector reused by
    chunk_hash and only the changed ones sent to `embed`, then one swap makes them current and retires the old
    rows with expire_at. Holds rows with no version at all (4.1's Document AI chunks on a lane older than the
    ledger): left alone, never written twice. Holds nothing: the first version, written current.
    `embed(texts) -> vectors` is the notebook's batched text-embedding-005 call. Returns {slug: chunks written}
    and prints one line per version event."""
    from google.cloud import firestore
    from google.cloud.firestore_v1.vector import Vector
    evals_dir = evals_dir or find_kit()
    counts = {}
    for doc in load_documents(tenant, evals_dir, project_id):
        key = f"{tenant}_{doc['sha256']}"
        held = rows_of(db, tenant, doc["source_uri"], collection)
        live = [r for r in held if r["current"] is not False and not r.get("staged")]
        counts[doc["slug"]] = 0
        if any(r["doc_key"] == key for r in live) or (held and not any(r["doc_key"] for r in held)):
            continue                                             # this version is current, or the rows predate versions
        if any(r["doc_key"] == key and r["current"] is False for r in held):
            n = swap_versions(db, held, key, retention_days, doc.get("effective_from"))
            record_version(db, tenant, doc["source_uri"], key, doc["sha256"], n["activated"], n["activated"], 0,
                           n["retired"], doc.get("effective_from"))
            print(f"  {doc['slug']}: reactivated {n['activated']} chunks, retired {n['retired']}, embedded 0 (the undo)")
            continue
        chunks = chunk_document(doc, tenant, doc["sha256"][:8] if live else "")
        if not chunks:
            continue
        by_hash = {r["chunk_hash"]: r["embedding"] for r in live
                   if r["chunk_hash"] and r["embedding"] is not None
                   and r["embedding_model"] == EMBEDDING_MODEL and str(r["embedding_version"]) == EMBEDDING_VERSION}
        vectors = [by_hash.get(c["chunk_hash"]) for c in chunks]           # the carry-over: reused by hash
        misses = [c["text"] for c, v in zip(chunks, vectors) if v is None]
        fresh = []
        for texts in embed_batches(misses):                                  # 250 texts AND 20,000 tokens per request
            fresh += embed(texts)
        it = iter(fresh)
        vectors = [list(v) if v is not None else next(it) for v in vectors]
        batch, n = db.batch(), 0
        for c, v in zip(chunks, vectors):
            row = {**c, "doc_key": key, "current": not live, "embedding": Vector(v),
                   "embedding_model": EMBEDDING_MODEL, "embedding_version": EMBEDDING_VERSION,
                   "schema_version": SCHEMA_VERSION, "indexed_at": firestore.SERVER_TIMESTAMP,
                   "processed_at": firestore.SERVER_TIMESTAMP}
            if doc.get("effective_from"):
                row["effective_from"] = doc["effective_from"]
            if live:
                row["staged"] = True                                 # a re-issue lands invisible; the swap makes it current
            batch.set(db.collection(collection).document(c["chunk_id"]), row)
            n += 1
            if n % 400 == 0:                                         # a Firestore batch holds 500 writes
                batch.commit()
                batch = db.batch()
        batch.commit()
        retired = 0
        if live:
            retired = swap_versions(db, rows_of(db, tenant, doc["source_uri"], collection), key, retention_days,
                                    doc.get("effective_from"))["retired"]
            print(f"  {doc['slug']}: revision {doc['sha256'][:8]}: {len(chunks) - len(misses)} chunks reused by hash, "
                  f"{len(misses)} embedded, {retired} retired")
        record_version(db, tenant, doc["source_uri"], key, doc["sha256"], len(chunks), len(chunks) - len(misses),
                       len(misses), retired, doc.get("effective_from"))
        counts[doc["slug"]] = len(chunks)
    return counts


### Wait for the indexes to build (first run only)

In [ ]:
# Block until the chunks indexes are READY (first run only, ~2-5 min).
# find_nearest raises FAILED_PRECONDITION until then. Safe to re-run any time.
print('Waiting for the chunks indexes to reach READY...')
for _ in range(40):  # up to ~10 min
    _rows = subprocess.run(
        ['gcloud', 'firestore', 'indexes', 'composite', 'list',
         '--project', PROJECT_ID, '--format=value(name,state)'],
        capture_output=True, text=True).stdout.lower()
    _rag = [ln for ln in _rows.splitlines() if 'chunks' in ln]
    if _rag and all('creating' not in ln for ln in _rag) and any('ready' in ln for ln in _rag):
        print('chunks indexes READY.')
        break
    time.sleep(15)
else:
    print('Still building after ~10 min. Wait a bit, then re-run THIS cell.')

## Cell 1: The context window is a budget

`TokenBudget` names every line of a DocuMind request. The four moves of context engineering map onto them: **write** (cache the stable tenant pack), **select** (retrieve and rerank), **compress** (pack to budget; summarise history in 8.5), **isolate** (one tenant per request).

In [ ]:
def count_tokens(text: str, model: str = 'gemini-3.6-flash') -> int:
    """Exact count, one API call. Use estimate_tokens() inside tight loops."""
    return gen_client.models.count_tokens(model=model, contents=text).total_tokens

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)   # ~4 chars per token for English; verify with count_tokens

@dataclass
class TokenBudget:
    system: int = 1_500
    tenant_pack: int = 40_000   # stable prefix -> cached (write). PRODUCTION scale: this
                                #   notebook's ACME pack is ~5k tokens (see the pack cell).
    chunks: int = 6_000         # retrieved evidence -> packed most-relevant-first (select)
    history: int = 2_000        # rolling summary; 8.5 adds the summarise node (compress)
    answer: int = 2_000         # reserved for the model's output

    @property
    def input_total(self) -> int:
        return self.system + self.tenant_pack + self.chunks + self.history

BUDGET = TokenBudget()

RAG_SYSTEM = """You are DocuMind, the ACME policy assistant.
Answer ONLY from the provided context. Cite every claim as [Source N].
If the context is insufficient, say so and set answerable=false.
A quote is the clause that answers - at most twenty-five words, never a whole section."""
# The last rule is the lane's own (generator.py, rule 5): a citation quote over 200 characters fails
# the contract and the SDK hands back None for the whole draft - on statutes, constantly (4.8, F21).

print(f'Input budget {BUDGET.input_total:,} tokens + {BUDGET.answer:,} reserved for the answer')
print('System prompt uses', count_tokens(RAG_SYSTEM), 'of', BUDGET.system)

## Cell 2: Hybrid retrieval — dense + sparse, fused with RRF

Dense search finds meaning; BM25 finds strings. Policy codes like `EXP-12` are strings. Reciprocal Rank Fusion merges the two rankings without comparing incompatible scores. `alpha` is the same knob Vector Search exposes as `rrf_ranking_alpha`.

In [ ]:
def embed_query(q: str) -> list[float]:
    return emb_client.models.embed_content(
        model='text-embedding-005', contents=q,
        config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY', output_dimensionality=768)
    ).embeddings[0].values

def load_tenant_chunks(tenant: str) -> list[dict]:
    """The tenant's CURRENT chunks, one version per source (the ledger, 12.5): a retired row is never a candidate,
    and when two versions of one document are both current for a moment (the swap writes a long document in more
    than one batch) the newest wins - the same guard rag-api's retriever applies."""
    rows, newest = [], {}
    for d in db.collection('chunks').where(filter=FieldFilter('tenant_id', '==', tenant)).stream():
        x = d.to_dict(); x.pop('embedding', None); x['chunk_id'] = d.id
        if x.get('current') is False:
            continue
        rows.append(x)
        at = x.get('reactivated_at') or x.get('indexed_at')
        if x.get('doc_key') and at is not None and (x['source_uri'] not in newest or at > newest[x['source_uri']][1]):
            newest[x['source_uri']] = (x['doc_key'], at)
    return [x for x in rows if not (x.get('doc_key') and x['source_uri'] in newest and x['doc_key'] != newest[x['source_uri']][0])]

def dense_search(q: str, tenant: str, k: int = 20) -> list[str]:
    """Firestore find_nearest with a tenant PRE-filter. Returns chunk_ids in rank order."""
    docs = (db.collection('chunks')
              .where(filter=FieldFilter('tenant_id', '==', tenant))
              .where(filter=FieldFilter('current', '==', True))      # the ledger's pre-filter: the second vector index
              .find_nearest(vector_field='embedding', query_vector=Vector(embed_query(q)),
                            distance_measure=DistanceMeasure.COSINE, limit=k,
                            distance_result_field='vector_distance')
              .get())
    return [d.id for d in docs]

def _tok(s: str) -> list[str]:
    return re.findall(r'[a-z0-9\-]+', s.lower())

def sparse_search(q: str, corpus: list[dict], k: int = 20) -> list[str]:
    """BM25 over the tenant's chunks, in-process. Production lane: Vector Search sparse vectors."""
    bm25 = BM25Okapi([_tok(c['text']) for c in corpus])
    scores = bm25.get_scores(_tok(q))
    order = sorted(range(len(corpus)), key=lambda i: -scores[i])
    return [corpus[i]['chunk_id'] for i in order[:k] if scores[i] > 0]

def rrf_fuse(dense_ids: list[str], sparse_ids: list[str], alpha: float = 0.5, k: int = 60):
    """Reciprocal Rank Fusion. alpha=1 -> dense only, 0 -> sparse only.
    Same knob as Vector Search HybridQuery(rrf_ranking_alpha=...). The lane's retriever
    (services/rag-api/hybrid.py) calls this with alpha=0.7: dense leads, BM25 rescues the codes."""
    fused = {}
    if alpha > 0:                      # alpha=0 means sparse only: contribute nothing from dense
        for rank, cid in enumerate(dense_ids):
            fused[cid] = fused.get(cid, 0) + alpha / (k + rank + 1)
    if alpha < 1:                      # alpha=1 means dense only
        for rank, cid in enumerate(sparse_ids):
            fused[cid] = fused.get(cid, 0) + (1 - alpha) / (k + rank + 1)
    return sorted(fused.items(), key=lambda kv: -kv[1])

_CORPUS_CACHE: dict = {}

def tenant_corpus(tenant: str) -> tuple:
    """Load (and remember) one tenant's chunks. Per tenant, never a shared global."""
    if tenant not in _CORPUS_CACHE:
        rows = load_tenant_chunks(tenant)
        _CORPUS_CACHE[tenant] = (rows, {c['chunk_id']: c for c in rows})
    return _CORPUS_CACHE[tenant]

CORPUS, BY_ID = tenant_corpus(TENANT)

def hybrid_retrieve(q: str, tenant: str, k: int = 20, alpha: float = 0.5) -> list[dict]:
    corpus, by_id = tenant_corpus(tenant)      # a different tenant gets ITS OWN corpus
    d, s = dense_search(q, tenant, k), sparse_search(q, corpus, k)
    out = []
    for cid, score in rrf_fuse(d, s, alpha)[:k]:
        if cid not in by_id:                   # dense hit outside this tenant's loaded set
            continue
        c = dict(by_id[cid]); c['rrf'] = round(score, 4)
        found = [n for n, hit in (('dense', cid in d), ('sparse', cid in s)) if hit]
        c['found_by'] = '+'.join(found)
        out.append(c)
    return out

q = 'What is the per-trip reimbursement cap under EXP-12?'
print('dense :', dense_search(q, TENANT, 5))
print('sparse:', sparse_search(q, CORPUS, 5))
for c in hybrid_retrieve(q, TENANT)[:5]:
    print(f"  {c['rrf']:.4f}  {c['found_by']:13}  {c.get('section') or c['chunk_id']}")

## Cell 3: Rerank 20 → 5 with the Rank API

`semantic-ranker-fast-004` is a cross-encoder: it reads the query and each candidate together, so it can tell *probation notice* from *notice period* in a way cosine distance cannot. It is served only from `global`, has a 1,024-token window per record, and is priced per request.

In [ ]:
_ranker = discoveryengine.RankServiceClient()
_ranking_config = _ranker.ranking_config_path(
    project=PROJECT_ID, location='global', ranking_config='default_ranking_config')

def rerank(q: str, chunks: list[dict], top_n: int = 5,
           model: str = 'semantic-ranker-fast-004') -> list[dict]:
    """Cross-encoder rerank of the fused candidates. 1,024-token window per record; served from global."""
    if not chunks:
        return chunks
    records = [discoveryengine.RankingRecord(
                   id=str(i), title=c.get('section', c.get('source_uri', 'chunk')),
                   content=c['text'][:4000])
               for i, c in enumerate(chunks)]
    resp = _ranker.rank(request=discoveryengine.RankRequest(
        ranking_config=_ranking_config, model=model, top_n=top_n, query=q, records=records))
    out = []
    for r in resp.records:
        c = dict(chunks[int(r.id)]); c['rerank_score'] = round(r.score, 3); out.append(c)
    return out

q = 'If I resign during probation versus after confirmation, what notice applies and can I encash leave?'
cands = hybrid_retrieve(q, TENANT, k=20)
t0 = time.time(); top5 = rerank(q, cands, top_n=5); ms = (time.time() - t0) * 1000
print(f'reranked {len(cands)} -> {len(top5)} in {ms:.0f} ms')
for c in top5:
    print(f"  {c['rerank_score']:.3f}  {c.get('section', c['source_uri'])}")

## Cell 4: Pack most-relevant-first, inside the budget

`pack_chunks()` keeps `source_uri`, page and section in every `[Source N]` header so citations survive packing. Watch what a budget sized for two of the five chunks drops.

In [ ]:
def pack_chunks(chunks: list[dict], budget_tokens: int):
    """Most-relevant-first packing. Keeps source_uri/page/section so every [Source N] still resolves."""
    packed, dropped, parts, used = [], [], [], 0
    for c in chunks:
        block = _block(len(packed) + 1, c)
        need = estimate_tokens(block)
        if used + need > budget_tokens:
            dropped.append(c); continue
        used += need; packed.append(c); parts.append(block)
    context = '\n\n'.join(parts)
    exact = count_tokens(context) if context else 0
    while exact > budget_tokens and packed:   # the estimate was optimistic: keep trimming
        dropped.insert(0, packed.pop()); parts.pop()
        context = '\n\n'.join(parts)
        exact = count_tokens(context) if context else 0
    return context, packed, dropped

def _block(n: int, c: dict) -> str:
    label = c.get('section') or c.get('source_uri', 'chunk').rsplit('/', 1)[-1]   # a statute window has no section
    page = f", p.{c['page_start']}" if c.get('page_start') else ''             # a .md upload has no page
    return f"[Source {n}] ({c['source_uri'].rsplit('/', 1)[-1]}{page}, {label})\n{c['text']}"

def _label(c: dict) -> str:
    return c.get('section') or c.get('chunk_id') or c.get('source_uri', 'chunk').rsplit('/', 1)[-1]

def small_budget(chunks: list[dict]) -> int:
    """A budget that fits the top TWO of these chunks and not the third - whatever their size.
    The failure this lesson ends on has to be stated in chunks, not in tokens: the loader's
    handbook clauses are ~80 tokens each, the lane's windows (services/ingest/main.py: 2,000
    characters) ~500 each. A fixed 150 packs everything on one corpus and nothing on the other."""
    needs = [estimate_tokens(_block(i, c)) for i, c in enumerate(chunks, 1)]
    return sum(needs[:2]) + (needs[2] // 2 if len(needs) > 2 else 0)

ctx, packed, dropped = pack_chunks(top5, BUDGET.chunks)
print(f'6,000-token budget: packed {len(packed)}, dropped {len(dropped)}')
SMALL_BUDGET = small_budget(top5)
ctx_small, packed_small, dropped_small = pack_chunks(top5, SMALL_BUDGET)
print(f'  {SMALL_BUDGET}-token budget (two of the five fit): packed {len(packed_small)}, dropped',
      [_label(c) for c in dropped_small])


## Cell 5: Structured generation (no sampling parameters)

`gemini-3.6-flash` ignores `temperature`, `top_p` and `top_k` and rejects penalty parameters. Use `thinking_level` and `max_output_tokens` instead.

In [ ]:
# THE answer contract - copied verbatim from deploy/shared/documind_schemas.py at build time,
# the same text 3.2, 4.2 and the rag-api service carry. The model is asked for a ModelDraft
# (it cites by the [Source N] it saw); resolve() turns that into a RAGAnswer whose Citations
# carry the chunk ids, pages and scores the model never knew.
class Citation(BaseModel):
    """One passage a caller can open. Text by default; a figure, a table or a video segment
    when the chunk is one - the four extra fields are optional and additive, so every citation
    written before Module 9 comes through unchanged."""

    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)
    kind: Literal["text", "figure", "table", "segment"] = "text"
    media_url: Optional[str] = None            # a signed URL for figure / table / segment
    start: Optional[float] = None              # seconds, for a video or audio segment
    end: Optional[float] = None


class RAGAnswer(BaseModel):
    """The contract. Modules 3 to 13 all pass this shape along."""

    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool


class DraftCitation(BaseModel):
    """What the model cites: the [Source N] number it saw, and the words it is relying on."""

    source: int = Field(ge=1, description="1-based [Source N] in the context")
    quote: str = Field(max_length=200, description="Exact words from that source")


class ModelDraft(BaseModel):
    """What the model is asked for. Use this as response_schema; resolve() turns it into a
    RAGAnswer. Asking the model for chunk ids or scores directly invites it to invent them."""

    answer: str
    citations: List[DraftCitation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool

def resolve(draft: ModelDraft, packed: List[dict]) -> RAGAnswer:
    """Turn the model's [Source N] citations into Citations, against the chunks it SAW.

    `packed` must be the list the context was built from - after the token budget dropped
    anything - not the list retrieval returned. Indexing into the pre-budget list shifts every
    citation after a dropped chunk by one, silently, which is exactly the bug rag-api had.

    An out-of-range index is dropped, not raised: the model's answer is still worth returning,
    and a citation to a source that was not in the context is not a citation.
    """
    cits: List[Citation] = []
    for d in draft.citations:
        if not 1 <= d.source <= len(packed):
            continue
        c = packed[d.source - 1]
        cits.append(Citation(
            chunk_id=str(c.get("chunk_id") or c.get("id") or ""),
            source_uri=c.get("source_uri", ""),
            page=c.get("page_start") or c.get("page"),
            quote=(d.quote or c.get("text", ""))[:500],
            score=float(min(1.0, max(0.0, c.get("rerank_score") or c.get("score") or 0.0))),
            kind=c.get("kind", "text"),
            media_url=c.get("media_url"),
            start=c.get("start"),
            end=c.get("end"),
        ))
    return RAGAnswer(answer=draft.answer, citations=cits,
                     confidence=draft.confidence, answerable=draft.answerable)

def gen_config(**extra) -> types.GenerateContentConfig:
    # gemini-3.6-flash ignores temperature/top_p/top_k and rejects penalties: pass none of them.
    return types.GenerateContentConfig(
        response_mime_type='application/json', response_schema=ModelDraft,
        thinking_config=types.ThinkingConfig(thinking_level='low'),
        max_output_tokens=BUDGET.answer, **extra)

def generate(question: str, context: str, packed: list[dict],
             cache_name: str | None = None, cache_client=None, pack_inline: str = ''):
    """cache_name  -> the stable prefix (system prompt + tenant pack) comes from the cache.
       pack_inline -> the uncached baseline: the same prefix is sent in full on every call."""
    body = f'Context:\n{context}\n\nQuestion: {question}'
    if cache_name:
        r = (cache_client or gen_client).models.generate_content(
            model='gemini-3.6-flash', contents=body, config=gen_config(cached_content=cache_name))
    else:
        r = gen_client.models.generate_content(
            model='gemini-3.6-flash', contents=[pack_inline, body] if pack_inline else body,
            config=gen_config(system_instruction=RAG_SYSTEM))
    draft = draft_of(r, ModelDraft)   # None from the SDK is a long quote or a cut-off, never a refusal (4.8, F21)
    # resolve() against `packed` - the post-budget list the context was built from - never the
    # reranked list: a dropped chunk would shift every later [Source N] by one, silently.
    return resolve(draft, packed), r.usage_metadata

print('answer contract (ModelDraft -> resolve() -> RAGAnswer) + generate() ready')

## Cell 6: Build the tenant pack (the stable prefix)

Glossary + FAQ + policy index for tenant `acme`, stating what the corpus states — the handbook's clauses and the statutes' figures — so the cached prefix and the retrieved chunks can never contradict each other. Explicit caching on the Gemini 3 family needs **at least 4,096 tokens**, so a synthetic revision log tops the pack up.

In [ ]:
GLOSSARY = [
    # The handbook's terms (deploy/evals/corpus/acme/hr_policy_2026.md) and the statutes' - the
    # same figures the chunks carry, so the cached prefix never argues with a retrieved clause.
    ('EL', 'Earned leave: accrues at 1.75 days per completed month (LV-01); encashed on exit under LV-07'),
    ('Encashment', 'Converting unused earned leave to pay at basic pay on exit, capped at 45 days (LV-07)'),
    ('Basic pay', 'The pay component earned leave is encashed at'),
    ('Form 16', 'Annual TDS certificate, issued by 15 June for the preceding financial year (PR-05)'),
    ('Notice period', 'From an acknowledged resignation to the last day: 60 days at E3 and above (NP-03), 15 days on probation (PB-02)'),
    ('Probation', 'Six months at grade E2, extendable once by up to three months with written reasons (PB-02)'),
    ('Per-trip cap', 'Rs 40,000 of domestic travel reimbursed against original receipts (EXP-12); above it, the function head approves before travel'),
    ('Function head', 'Approves purchases up to Rs 2,00,000; the CFO approves above that (FIN-02)'),
    ('Remote work', 'Up to eight days a month with manager consent; from outside India only with prior tax clearance (WFH-01)'),
    ('Removable media', 'USB mass-storage is blocked on all company laptops, no exception for contractors (IT-SEC-04)'),
    ('Access review', 'Quarterly; an account unused for 45 days is disabled and re-approved to restore (SEC-09)'),
    ('MSA', 'Master services agreement with DocuMind Technologies: 24 months, terminable for convenience on 90 days notice (MSA-04)'),
    ('Gratuity', 'Statutory: after five years of continuous service, at fifteen days wages per completed year (Payment of Gratuity Act, 1972, s. 4)'),
    ('Maternity benefit', 'Statutory: twenty-six weeks since the 2017 amendment; the principal Act of 1961 said twelve'),
    ('Bonus', 'Statutory: 8.33 per cent minimum, twenty per cent maximum, after thirty working days in the year (Payment of Bonus Act, 1965)'),
    ('Overtime', 'Statutory: at least twice the normal rate of wages (Code on Wages, 2019, s. 14)'),
    ('POSH', 'Prevention of sexual harassment: the Internal Committee under the POSH Act, 2013 (a scanned Gazette in the corpus)'),
]
FAQ = [
    ('Can I encash leave while serving notice?', 'No. LV-07: leave cannot be encashed during probation and cannot be used to shorten notice.'),
    ('How much earned leave can I carry forward?', 'Up to 30 days; anything above lapses on 31 December (LV-01).'),
    ('What is my notice period as an E3?', '60 days under NP-03; 15 days for either side while on probation (PB-02).'),
    ('Can my unused leave shorten my notice?', 'No. NP-03: unused earned leave may not be set off against the notice period.'),
    ('What is the cap on a domestic trip?', 'Rs 40,000 per trip against original receipts (EXP-12).'),
    ('Who approves a Rs 3,00,000 purchase?', 'The CFO; the function head approves up to Rs 2,00,000 (FIN-02).'),
    ('When is Form 16 issued?', 'By 15 June each year (PR-05).'),
    ('How many remote days can I take?', 'Up to eight a month with manager consent (WFH-01).'),
    ('Is a USB drive allowed?', 'No. USB mass storage is blocked on all company laptops, with no exception for contractors (IT-SEC-04).'),
    ('After how many years is gratuity payable?', 'Five years of continuous service, at fifteen days wages per completed year (Payment of Gratuity Act, 1972, s. 4).'),
    ('How many weeks of maternity benefit does the law provide?', 'Twenty-six weeks, since the Maternity Benefit (Amendment) Act, 2017.'),
    ('What is the minimum bonus?', '8.33 per cent of the wages earned in the accounting year (Payment of Bonus Act, 1965, s. 10).'),
]

def build_tenant_pack(policies, min_tokens: int = 4_300) -> str:
    """Stable per-tenant prefix: glossary + FAQ + policy index.
    Must clear the 4,096-token minimum for explicit caching on the Gemini 3 family."""
    lines = ['ACME EMPLOYEE HANDBOOK - GLOSSARY, FAQ AND POLICY INDEX (tenant: acme)', '', 'GLOSSARY']
    lines += [f'- {term}: {defn}' for term, defn in GLOSSARY]
    lines += ['', 'FREQUENTLY ASKED QUESTIONS'] + [f'Q: {q}\nA: {a}' for q, a in FAQ]
    lines += ['', 'POLICY INDEX'] + [f'- {code} {title} ({src}, p.{page})' for code, title, _, src, page in policies]
    pack = '\n'.join(lines)
    i = 0
    while estimate_tokens(pack) < int(min_tokens * 1.15):   # synthetic revision log fills the gap
        code, title, _, _, _ = policies[i % len(policies)]
        pack += (f'\nREVISION LOG {i + 1:03d}: {code} {title} reviewed on 2026-{(i % 12) + 1:02d}-15 '
                 f'by the policy owner; entitlements unchanged; next review due in 12 months.')
        i += 1
    while count_tokens(pack) < min_tokens:                    # exact check, top up if the estimate was high
        pack += '\nREVISION LOG: annual review completed; no change to any entitlement listed above.' * 20
    return pack

TENANT_PACK = build_tenant_pack(ACME_POLICIES)
PACK_TOKENS = count_tokens(TENANT_PACK)
print(f'tenant pack: {PACK_TOKENS:,} tokens (explicit-cache minimum on the Gemini 3 family: 4,096)')

## Cell 7: Explicit cache on `global`, recorded in Firestore `tenant_caches`

An explicit cache lives where generation does: `global` (Google's own Gemini 3 caching sample; CLAUDE.md, 2026-09-04). This cell creates it there, records it in Firestore under this lesson's own key - never `tenant_caches/acme`, the document the lane's API reads on every query - and runs uncached if the create fails. Note the TTL: explicit caches have **no maximum**, so a cache without a TTL bills storage forever.

In [ ]:
def create_tenant_cache(tenant: str, pack: str, ttl_s: int = 3600):
    cfg = types.CreateCachedContentConfig(
        display_name=f'documind-{tenant}',
        system_instruction=RAG_SYSTEM,
        contents=[types.Content(role='user', parts=[types.Part.from_text(text=pack)])],
        ttl=f'{ttl_s}s')   # the DEFAULT TTL is 60 minutes; no maximum is documented, so set one
    # The cache is created on `global`, where Gemini 3.x generation is served: a regional cache
    # could serve nothing (CLAUDE.md, 2026-09-04). If the create fails, the lesson runs the
    # UNCACHED path - the same prefix sent in full on every call - and says so.
    try:
        cache, where = gen_client.caches.create(model='gemini-3.6-flash', config=cfg), 'global'
    except Exception as e:
        print('cache create on global failed:', str(e)[:140])
        print('-> running the UNCACHED path: the same prefix is sent in full on every call.')
        print('   Keep the error: quota, a missing permission and a model name each read differently.')
        return None, None
    # NOT tenant_caches/{tenant}: that document is the one the lane's API reads for every /v1/query
    # (services/rag-api/cache_manager.py). A notebook that wrote it would swap the live service's
    # prompt for this lesson's pack until the TTL ran out - so this lesson keeps its own record.
    cache_doc(tenant).set({
        'tenant_id': tenant, 'cache_name': cache.name, 'location': where, 'model': 'gemini-3.6-flash',
        'tokens': cache.usage_metadata.total_token_count, 'expire_time': cache.expire_time,
        'created_at': firestore.SERVER_TIMESTAMP})
    return cache.name, where

def cache_doc(tenant: str):
    return db.collection('tenant_caches').document(f'{tenant}-lesson-4.5')

def get_or_create_tenant_cache(tenant: str, pack: str):
    doc = cache_doc(tenant).get()
    if doc.exists:
        rec = doc.to_dict()
        if rec['expire_time'] > dt.datetime.now(dt.timezone.utc) + dt.timedelta(minutes=2):
            return rec['cache_name'], rec['location']
    return create_tenant_cache(tenant, pack)

CACHE_NAME, CACHE_WHERE = get_or_create_tenant_cache(TENANT, TENANT_PACK)
print('cache:', CACHE_NAME or 'NONE (uncached path)', '| endpoint:', CACHE_WHERE or 'n/a')

## Cell 8: Cost side by side — uncached vs cached, intro vs 2027 standard

The uncached baseline sends the same prefix inline on every call. `cached_content_token_count` shows how much of the prompt was billed at the cached rate (90% discount).

In [ ]:
PRICING = {  # USD per 1M tokens, gemini-3.6-flash on Vertex AI (re-verify on the pricing page)
    'intro (to 2026-12-31)':      {'input': 0.75, 'output': 3.75, 'cached': 0.075, 'storage_per_hour': 0.50},
    'standard (from 2027-01-01)': {'input': 1.50, 'output': 7.50, 'cached': 0.15,  'storage_per_hour': 1.00},
}

def query_cost(usage, rate: str = 'intro (to 2026-12-31)') -> dict:
    p = PRICING[rate]
    cached = usage.cached_content_token_count or 0
    fresh = usage.prompt_token_count - cached
    out = usage.candidates_token_count + (usage.thoughts_token_count or 0)   # thinking bills as output
    usd = fresh / 1e6 * p['input'] + cached / 1e6 * p['cached'] + out / 1e6 * p['output']
    return {'fresh_in': fresh, 'cached_in': cached, 'out': out, 'usd': usd, 'inr': usd * USD_INR}

q = 'If I resign during probation versus after confirmation, what notice applies and can I encash leave?'
top5 = rerank(q, hybrid_retrieve(q, TENANT), 5)
ctx, packed, _ = pack_chunks(top5, BUDGET.chunks)
res_a, use_a = generate(q, ctx, packed, pack_inline=TENANT_PACK)                       # uncached baseline
res_b, use_b = generate(q, ctx, packed, cache_name=CACHE_NAME)                          # cached prefix
for label, u in (('uncached', use_a), ('cached', use_b)):
    for rate in PRICING:
        c = query_cost(u, rate)
        print(f"{label:9} {rate:28} fresh={c['fresh_in']:6,} cached={c['cached_in']:6,} "
              f"out={c['out']:4,}  ${c['usd']:.5f}  Rs {c['inr']:.3f}")
print('\ncached answer:', res_b.answer[:220])
print('citations:', [(c.chunk_id, round(c.score, 2)) for c in res_b.citations])

# Storage: a 40k-token production pack at $0.50 per 1M token-hours = $0.02/hour. It pays for
# itself after roughly one query per hour. This notebook's pack is ~5k tokens, so its storage
# is ~$0.0025/hour. A cache left behind bills storage until its TTL expires - delete it.

## Cell 9: `ask_documind_v2()` — the whole pipeline with a JSON log line

The log fields are the ones rag-api writes to stdout so BigQuery `tenant_daily` (12.6) can report `cached_tokens` and `retrieval_mode` per tenant.

In [ ]:
def ask_documind_v2(question: str, tenant: str = TENANT, use_cache: bool = True,
                    chunk_budget: int | None = None, alpha: float = 0.5) -> dict:
    t0 = time.time()
    cands = hybrid_retrieve(question, tenant, k=20, alpha=alpha)          # 20 of the tenant's ~1,600 chunks
    top = rerank(question, cands, top_n=5)                                 # select: cross-encoder
    ctx, packed, dropped = pack_chunks(top, chunk_budget or BUDGET.chunks)  # compress: budgeted packing
    if not packed:
        return {'answer': 'No relevant context.', 'citations': [],
                'log': {'tenant_id': tenant, 'retrieval_mode': 'hybrid', 'candidates': len(cands),
                        'packed': 0, 'dropped': len(dropped), 'cached_tokens': 0, 'prompt_tokens': 0,
                        'cost_inr': 0.0, 'latency_ms': int((time.time() - t0) * 1000),
                        'citations': 0, 'confidence': 'low', 'answerable': False}}
    if use_cache and CACHE_NAME:                                           # write: cached stable prefix
        res, usage = generate(question, ctx, packed, cache_name=CACHE_NAME)
    else:                                                                  # no cache: full prefix inline
        res, usage = generate(question, ctx, packed, pack_inline=TENANT_PACK)
    cost = query_cost(usage)
    log = {  # the same fields rag-api writes to its JSON log -> BigQuery tenant_daily (12.6)
        'tenant_id': tenant, 'retrieval_mode': 'hybrid', 'candidates': len(cands), 'packed': len(packed),
        'dropped': len(dropped), 'cached_tokens': cost['cached_in'], 'prompt_tokens': usage.prompt_token_count,
        'cost_inr': round(cost['inr'], 4), 'latency_ms': int((time.time() - t0) * 1000),
        'citations': len(res.citations), 'confidence': res.confidence, 'answerable': res.answerable}
    return {'answer': res.answer, 'citations': [c.model_dump() for c in res.citations], 'log': log}

for question in [
        'What is the per-trip reimbursement cap under EXP-12?',
        'If I resign during probation versus after confirmation, what notice applies and can I encash leave?',
        'After how many years of continuous service is gratuity payable, and at what rate?']:   # a real Act, a real page
    r = ask_documind_v2(question)
    print(json.dumps(r['log']))
    print('  ->', r['answer'][:160], '\n')

## Cell 10: What breaks when the budget fits two chunks of five

This is the forward hook to 4.7: the lost citation becomes a failing row in `golden.jsonl`, not a story.

In [ ]:
# The forward hook to 4.7: this becomes a failing eval row, not a story. On the lane the top five
# are 2,000-character windows of the handbook, not single clauses, so the budget is stated as
# 'two of the five' (small_budget) rather than a number that means nothing on another corpus.
q = 'If I resign during probation versus after confirmation, what notice applies and can I encash leave?'
full = ask_documind_v2(q, chunk_budget=BUDGET.chunks)
tiny = ask_documind_v2(q, chunk_budget=SMALL_BUDGET)   # two of the five reranked chunks fit, the third does not
print('6,000-token budget:', full['log']['packed'], 'chunks packed,', full['log']['citations'],
      'citations, answerable =', full['log']['answerable'])
print(f'  {SMALL_BUDGET}-token budget:', tiny['log']['packed'], 'chunks packed,', tiny['log']['citations'],
      'citations, answerable =', tiny['log']['answerable'])
print('Compare the citations: a dropped clause is a lost citation, whichever chunk carried it. Budgets are a quality knob.')

## Production lane (badge: requires 12.1–12.5 for rag-api, 12.8 for documind-chat)

The same three ideas ship as `context_budget.py`, `hybrid.py` and `cache_manager.py` inside `deploy/services/rag-api/`. In production, Vector Search fuses dense and sparse vectors server-side and the tenant restrict is part of the query. This cell is reference code (needs `google-cloud-aiplatform` and a deployed index); it is shown, not run.

```python
# Production lane (requires 12.1-12.5 for rag-api). Vector Search fuses dense + sparse server-side.
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import HybridQuery, Namespace

def hybrid_find_neighbors(index_endpoint, deployed_index_id, dense_vec, sparse_vals, sparse_dims,
                          tenant_id, k=20, alpha=0.5):
    q = HybridQuery(dense_embedding=dense_vec, sparse_embedding_values=sparse_vals,
                    sparse_embedding_dimensions=sparse_dims, rrf_ranking_alpha=alpha)
    return index_endpoint.find_neighbors(
        deployed_index_id=deployed_index_id, queries=[q], num_neighbors=k,
        filter=[Namespace(name='tenant_id', allow_tokens=[tenant_id])])   # tenant restrict, never post-filter
```

## Cleanup — run this before you close the notebook

In [ ]:
# CLEANUP. The default TTL is 60 minutes and no maximum is documented, so a cache with a long
# TTL bills storage every hour until it expires. Delete it explicitly.
def delete_tenant_cache(tenant: str):
    doc = cache_doc(tenant).get()
    if not doc.exists:
        print('no cache record'); return
    rec = doc.to_dict()
    cl = gen_client if rec['location'] == 'global' else emb_client
    try:
        cl.caches.delete(name=rec['cache_name']); print('deleted', rec['cache_name'])
    except Exception as e:
        print('delete failed (already expired?)', str(e)[:80])
    cache_doc(tenant).delete()          # only what this lesson wrote - never the lane's record

delete_tenant_cache(TENANT)

## ✅ Lesson 4.5 Complete!

- ✅ `TokenBudget`: system · tenant pack · chunks · history · answer
- ✅ Hybrid retrieval: Firestore dense + BM25, fused with RRF (`alpha`)
- ✅ Rank API `semantic-ranker-fast-004`, top 5 of 20 candidates, served from `global`
- ✅ `pack_chunks()`: most-relevant-first, metadata preserved, budget enforced
- ✅ Explicit cache ≥ 4,096 tokens on `global` (no regional path for Gemini 3.x), TTL 3600 s, this lesson's own `tenant_caches` record
- ✅ Dual-priced INR cost per query from `usage_metadata`
- ✅ `ask_documind_v2()` with the JSON log fields rag-api will emit

**Next: Lesson 4.6 — Graph RAG on Spanner Graph (multi-hop questions across documents). Then 4.7 turns today's small-budget failure into an eval gate, and 4.8 runs the gate against the live lane.**